# Google Trends quickstart

Run the Actor `tallydata/google-trends-interest-over-time` and plot the result. **The Actor is not published yet, so this will fail until it is.**

Unofficial. Not affiliated with or endorsed by Google.

Add your Apify token in Colab under **Secrets** (key icon) as `APIFY_TOKEN`. It's read from there, and never stored in this notebook.

Each keyword is charged per the Actor's Store pricing.

In [ ]:
!pip install -q apify-client pandas matplotlib

In [ ]:
import os

try:
    from google.colab import userdata

    token = userdata.get("APIFY_TOKEN")
except ImportError:
    token = os.environ["APIFY_TOKEN"]

In [ ]:
from apify_client import ApifyClient

ACTOR = "tallydata/google-trends-interest-over-time"
run_input = {"searchTerms": ["web scraping"], "geo": "US"}

client = ApifyClient(token)
run = client.actor(ACTOR).call(run_input=run_input)
items = list(client.dataset(run["defaultDatasetId"]).iterate_items())
items[0]["searchTerm"]

In [ ]:
import pandas as pd

timeline = items[0]["interestOverTime_timelineData"]
frame = pd.DataFrame(
    {
        "date": pd.to_datetime([int(p["time"]) for p in timeline], unit="s"),
        "interest": [p["value"][0] for p in timeline],
        "partial": [p.get("isPartial", False) for p in timeline],
    }
)
frame[~frame["partial"]].plot(x="date", y="interest", title=items[0]["searchTerm"]);